In [ ]:
import subprocess
from pydub import AudioSegment
import math



def extract_audio_from_video(video_path, audio_path):
  command = ["ffmpeg", "-i", video_path, "-vn", audio_path]
  subprocess.run(command)


def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder):
  track = AudioSegment.from_mp3(audio_path)
  
  chunk_len = chunk_size * 60 * 1000

  chunks = math.ceil(len(track) / chunk_len)

  for i in range(chunks):
    start_time = i * chunk_len
    end_time = (i + 1) * chunk_len

    chunk = track[start_time:end_time]

    chunk.export(f"{chunks_folder}/chunk_{i}.mp3", format="mp3")


In [ ]:
extract_audio_from_video("./files/video.mp4", "./files/audio.mp3")

In [5]:
cut_audio_in_chunks("./files/audio.mp3", 10, "./files/chunks")

In [12]:
import openai
import glob


def transcribe_chunks(chunk_folder, destination):
  files = glob.glob(f"{chunk_folder}/*.mp3")

  for file in files:
    
    # rb: reading as binary
    # a: append mode
    with open(file, "rb") as audio_file, open(destination, "a") as text_file:
      
      transript = openai.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file, 
        language="ko"
      )

      text_file.write(transript.text)


transcribe_chunks("./files/chunks", "./files/transcript.txt")